# 实验四（课堂演示）：神经图像 vs JPEG

**适用课程**：未来媒体互联网  
**演示时长**：约 10 分钟  
**运行环境**：Kaggle Notebook（PyTorch 预装）

## 演示目标

让学生直观对比 JPEG 和Neural两种编码方式的差异：
- JPEG：基于 DCT 变换，固定算法，会产生方块效应
- Neural：端到端训练的神经网络，自适应学习策略

核心展示：相近比下，JPEG 的方块效应 vs Neural的平滑重建。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io

print("PyTorch 已就绪 ✅")

In [ ]:
# 生成演示用测试图像：包含高频和低频区域
def create_demo_image(size=128):
    img = np.zeros((size, size), dtype=np.float32)
    # 棋盘格（高频）
    for i in range(0, size, 8):
        for j in range(0, size, 8):
            if (i//8 + j//8) % 2 == 0:
                img[i:i+8, j:j+8] = 0.9
            else:
                img[i:i+8, j:j+8] = 0.1
    # 平滑渐变（低频）
    for i in range(20, 60):
        img[i+60, 20:100] = 0.3 + (i-20)/40 * 0.6
    # 细线
    img[100:110, :] = 1.0
    return img

original = create_demo_image(128)

plt.figure(figsize=(4, 4))
plt.imshow(original, cmap='gray', vmin=0, vmax=1)
plt.title('Test Image\n（Checkerboard=HF|Gradient=LF|Line=Edge）')
plt.axis('off')
plt.show()

In [ ]:
# 简单的神经自编码器（卷积版本）
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # 编码器：到 1/4 尺寸
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 8, 3, stride=2, padding=1), nn.ReLU(),
        )
        # 解码器：恢复到原始尺寸
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(8, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid(),
        )
    
    def forward(self, x):
        return self.decoder(self.encoder(x))

# 创建并训练一个小模型来演示概念（实际用预训练模型更快）
# 这里用简单方式演示：用不同程度的降采样来模拟"Neural"
# 这样无需训练，直接展示概念

def neural_compress_simple(img_tensor, compression_level):
    """
    简化模拟Neural：通过降采样+上采样模拟
    实际Neural比这复杂得多，但视觉效果类似
    """
    factors = [1, 2, 4, 8]
    factor = factors[min(compression_level, 3)]
    if factor == 1:
        return img_tensor.clone()
    
    _, _, h, w = img_tensor.shape
    # 降采样
    small = F.interpolate(img_tensor, size=(h//factor, w//factor), mode='area')
    # 上采样
    restored = F.interpolate(small, size=(h, w), mode='bilinear', align_corners=False)
    return restored

print("Neural模拟器就绪 ✅")
print("注：实际Neural用端到端训练的网络，这里用采样模拟原理展示")

In [ ]:
# JPEG（量化造成的方块效应）
def jpeg_simulate(img_tensor, quality):
    """
    模拟 JPEG 效果：
    - 对 8x8 块做平均（模拟量化）
    - quality 越低，块越大
    """
    block_sizes = [1, 4, 8, 16]
    block = block_sizes[min(quality, 3)]
    if block == 1:
        return img_tensor.clone()
    
    result = img_tensor.clone()
    _, _, h, w = result.shape
    for i in range(0, h, block):
        for j in range(0, w, block):
            i_end = min(i + block, h)
            j_end = min(j + block, w)
            result[:, :, i:i_end, j:j_end] = result[:, :, i:i_end, j:j_end].mean()
    return result

print("JPEG器就绪 ✅")
print("注：真实 JPEG 的块效应来自 8x8 DCT 量化，这里用块平均近似展示")

In [ ]:
# 构建对比矩阵
img_t = torch.tensor(original).unsqueeze(0).unsqueeze(0).float()

levels = {'Lossless': 0, 'Light': 1, 'Medium': 2, 'Heavy': 3}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for j, (label, lv) in enumerate(levels.items()):
    # JPEG
    jpeg_result = jpeg_simulate(img_t, lv)
    axes[0, j].imshow(jpeg_result.squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[0, j].set_title(f'JPEG\n{label}', fontsize=11)
    axes[0, j].axis('off')
    
    # Neural模拟
    neural_result = neural_compress_simple(img_t, lv)
    axes[1, j].imshow(neural_result.squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[1, j].set_title(f'Neural\n{label}', fontsize=11)
    axes[1, j].axis('off')

plt.suptitle('JPEG vs Neural Compression\n（Top: JPEG | Bottom: Neural）',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("1. JPEG: visible blocks at medium+ compression (8x8 boundaries), checkerboard lost")
print("2. Neural: preserves structure but details blur (downsample-upsample smoothing)")
print("3. Real neural compression uses end-to-end trained CNN/Transformer, much better than this simulation")

## 课堂演示流程（10 分钟）

1. **0-2 min**：展示原始图像，强调"棋盘格=高频细节，渐变=低频平滑"
2. **2-5 min**：逐列对比 JPEG 和Neural，从Lossless到Heavy
3. **5-8 min**：聚焦"Medium"列——JPEG 大方块 vs Neural的模糊但连贯
4. **8-10 min**：讨论"为什么深度学习能做得更好"——自适应学习 vs 固定算法

## 课后延伸

完整版实验使用 CompressAI 预训练模型，在真实照片上对比 JPEG 和Neural的率失真曲线。